# Estudo de Caso 4 - Sistemas de Recomendação para Produtos da Amazon

Este notebook apresenta a implementação de dois sistemas de recomendação para produtos eletrônicos da Amazon, utilizando técnicas diferentes de aprendizado de máquina:
1. **Parte 1**: Sistema de recomendação baseado no algoritmo KNN (K-Nearest Neighbors)
2. **Parte 2**: Sistema de recomendação baseado em clusterização com K-Means

### Contexto
Como Cientista de Dados na Amazon, nossa tarefa é construir um sistema de recomendação para sugerir produtos aos clientes com base em suas avaliações anteriores. Utilizaremos um conjunto de dados contendo classificações de diferentes produtos eletrônicos.

### Objetivo
Extrair insights significativos dos dados e construir sistemas de recomendação que ajudem a recomendar produtos aos consumidores online, utilizando duas abordagens diferentes para comparação.

### Atributos do Dataset
- **userId**: identificador único para cada usuário
- **productId**: identificador único para cada produto
- **Rating**: avaliação dada pelo usuário ao produto 
- **Timestamp**: momento da avaliação (não será utilizado nesta análise)

## Importação de Bibliotecas e Carregamento dos Dados

In [ ]:
# Importando as bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
warnings.filterwarnings('ignore')

# Carregamento do dataset
df = pd.read_csv('../../data/databaseEletronicos.csv')

# Visualizando as primeiras linhas do dataset
df.head()

## Análise Exploratória dos Dados

Vamos examinar os dados para entender melhor sua estrutura, distribuição e possíveis desafios para criação dos sistemas de recomendação.

In [ ]:
# Informações gerais do dataset
print("=== Informações do Dataset ===")
df.info()

# Estatísticas descritivas
print("\n=== Estatísticas Descritivas ===")
print(df.describe())

# Verificação de valores ausentes
print("\n=== Valores Ausentes ===")
print(df.isnull().sum())

In [ ]:
# Distribuição das avaliações
plt.figure(figsize=(10, 6))
sns.histplot(df['Rating'], bins=10, kde=True)
plt.title('Distribuição das Avaliações', fontsize=15)
plt.xlabel('Avaliação', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.xticks(range(1, 6))
plt.axvline(df['Rating'].mean(), color='red', linestyle='--', label=f'Média: {df["Rating"].mean():.2f}')
plt.axvline(df['Rating'].median(), color='green', linestyle='-.', label=f'Mediana: {df["Rating"].median():.2f}')
plt.legend()
plt.show()

In [ ]:
# Análise do número de avaliações por usuário
user_ratings_count = df['userId'].value_counts()

print(f"Número total de usuários: {df['userId'].nunique()}")
print(f"Número médio de avaliações por usuário: {user_ratings_count.mean():.2f}")
print(f"Número máximo de avaliações por usuário: {user_ratings_count.max()}")
print(f"Número mínimo de avaliações por usuário: {user_ratings_count.min()}")

plt.figure(figsize=(12, 6))
sns.histplot(user_ratings_count, bins=50, kde=True)
plt.title('Distribuição do Número de Avaliações por Usuário', fontsize=15)
plt.xlabel('Número de Avaliações', fontsize=12)
plt.ylabel('Número de Usuários', fontsize=12)
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Análise do número de avaliações por produto
product_ratings_count = df['productId'].value_counts()

print(f"Número total de produtos: {df['productId'].nunique()}")
print(f"Número médio de avaliações por produto: {product_ratings_count.mean():.2f}")
print(f"Número máximo de avaliações por produto: {product_ratings_count.max()}")
print(f"Número mínimo de avaliações por produto: {product_ratings_count.min()}")

plt.figure(figsize=(12, 6))
sns.histplot(product_ratings_count, bins=50, kde=True)
plt.title('Distribuição do Número de Avaliações por Produto', fontsize=15)
plt.xlabel('Número de Avaliações', fontsize=12)
plt.ylabel('Número de Produtos', fontsize=12)
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## Pré-processamento dos Dados

Antes de implementar os algoritmos de recomendação, precisamos realizar algumas etapas de pré-processamento para preparar os dados.

In [ ]:
# Vamos filtrar para manter apenas usuários e produtos com um número mínimo de avaliações
# Isso ajuda a reduzir o problema de cold-start e melhora a qualidade das recomendações

# Definindo limiares
min_user_ratings = 10  # Usuários com pelo menos 10 avaliações
min_product_ratings = 5  # Produtos com pelo menos 5 avaliações

# Filtrando usuários com base no número mínimo de avaliações
users_to_keep = user_ratings_count[user_ratings_count >= min_user_ratings].index
df_filtered = df[df['userId'].isin(users_to_keep)]

# Filtrando produtos com base no número mínimo de avaliações
products_to_keep = product_ratings_count[product_ratings_count >= min_product_ratings].index
df_filtered = df_filtered[df_filtered['productId'].isin(products_to_keep)]

print(f"Dimensões do dataset original: {df.shape}")
print(f"Dimensões do dataset filtrado: {df_filtered.shape}")
print(f"Número de usuários após filtragem: {df_filtered['userId'].nunique()}")
print(f"Número de produtos após filtragem: {df_filtered['productId'].nunique()}")

In [ ]:
# Verificando se há duplicatas nas avaliações (mesmo usuário avaliando o mesmo produto mais de uma vez)
duplicates = df_filtered[df_filtered.duplicated(subset=['userId', 'productId'], keep=False)]
print(f"Número de avaliações duplicadas: {len(duplicates)}")

if len(duplicates) > 0:
    # No caso de duplicatas, vamos manter apenas a avaliação mais recente
    df_filtered = df_filtered.sort_values('Timestamp').drop_duplicates(subset=['userId', 'productId'], keep='last')
    print(f"Dimensões do dataset após remoção de duplicatas: {df_filtered.shape}")

# Parte 1: Sistema de Recomendação baseado em KNN

Nesta primeira parte, implementaremos um sistema de recomendação utilizando o algoritmo K-Nearest Neighbors (KNN). Este método se baseia na ideia de que produtos similares receberão avaliações similares dos mesmos usuários.

In [ ]:
# Criando a matriz de utilidade usuário-item (pivot table)
# Linhas representam produtos, colunas representam usuários, valores são as avaliações
ratings_matrix = df_filtered.pivot_table(values='Rating', index='productId', columns='userId')

# Visualizando a matriz de avaliações (primeiras linhas e colunas)
print("Dimensões da matriz de avaliações:", ratings_matrix.shape)
ratings_matrix.iloc[:5, :5]

In [ ]:
# Preenchendo valores ausentes com 0
# Isso significa que o usuário não avaliou o produto
ratings_matrix_filled = ratings_matrix.fillna(0)

# Convertendo a matriz para um formato esparso para economizar memória
# Isso é especialmente importante para conjuntos de dados grandes
ratings_sparse = csr_matrix(ratings_matrix_filled.values)

In [ ]:
# Implementando o algoritmo KNN para encontrar produtos similares
# Usaremos a distância de cosseno que é eficaz para dados de avaliação
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(ratings_sparse)

### Função de Recomendação com KNN

Agora vamos criar uma função que recomenda produtos com base na similaridade calculada pelo algoritmo KNN.

In [ ]:
# Função para recomendar produtos similares usando KNN
def recommend_products_knn(product_id, n_recommendations=5):
    """
    Recomenda produtos similares a um produto específico usando KNN.
    
    Parâmetros:
        product_id: ID do produto para o qual queremos recomendações
        n_recommendations: Número de recomendações a retornar
        
    Retorna:
        Lista de IDs de produtos recomendados e suas distâncias
    """
    
    # Verificar se o produto existe na matriz
    if product_id not in ratings_matrix.index:
        print(f"O produto {product_id} não está no conjunto de dados filtrado.")
        return None, None
    
    # Encontrar o índice do produto na matriz
    product_idx = ratings_matrix.index.get_loc(product_id)
    
    # Encontrar os k+1 produtos mais similares (incluindo o próprio produto)
    distances, indices = model_knn.kneighbors(
        ratings_matrix_filled.iloc[product_idx, :].values.reshape(1, -1),
        n_neighbors=n_recommendations+1
    )
    
    # Converter índices para IDs de produtos
    product_ids = [ratings_matrix.index[idx] for idx in indices.flatten()]
    
    # Remover o próprio produto da lista (primeira posição)
    product_ids = product_ids[1:]
    distances = distances.flatten()[1:]
    
    return product_ids, distances

In [ ]:
# Função para mostrar as recomendações e informações relevantes
def display_recommendations_knn(product_id, n_recommendations=5):
    """
    Exibe os produtos recomendados para um produto específico.
    """
    product_ids, distances = recommend_products_knn(product_id, n_recommendations)
    
    if product_ids is None:
        return
    
    # Obter informações sobre o produto de base
    base_product_ratings = df_filtered[df_filtered['productId'] == product_id]
    base_avg_rating = base_product_ratings['Rating'].mean()
    base_num_ratings = len(base_product_ratings)
    
    print(f"Recomendações para o produto {product_id}:")
    print(f"Avaliação média: {base_avg_rating:.2f} (baseado em {base_num_ratings} avaliações)\n")
    
    print("{:<15} {:<10} {:<15} {:<20}".format("Produto", "Distância", "Avaliação Média", "Número de Avaliações"))
    print("-" * 65)
    
    for i, (rec_id, distance) in enumerate(zip(product_ids, distances)):
        # Obter informações sobre o produto recomendado
        rec_ratings = df_filtered[df_filtered['productId'] == rec_id]
        avg_rating = rec_ratings['Rating'].mean()
        num_ratings = len(rec_ratings)
        
        print("{:<15} {:<10.4f} {:<15.2f} {:<20}".format(
            rec_id, distance, avg_rating, num_ratings
        ))

In [ ]:
# Encontrando os 5 produtos mais populares (com mais avaliações)
top_products = product_ratings_count.nlargest(5)
print("Os 5 produtos mais populares (com mais avaliações):")
for product_id, count in top_products.items():
    avg_rating = df[df['productId'] == product_id]['Rating'].mean()
    print(f"Produto {product_id}: {count} avaliações, nota média: {avg_rating:.2f}")

In [ ]:
# Testando o sistema de recomendação para um dos produtos mais populares
# Substituir pelo ID de um produto real do seu dataset
test_product_id = top_products.index[0]  # Usando o produto mais popular
display_recommendations_knn(test_product_id, n_recommendations=5)

In [ ]:
# Testando com alguns produtos adicionais
for i in range(1, 3):  # Testando o segundo e terceiro produtos mais populares
    test_product_id = top_products.index[i]
    print("\n" + "=" * 70)
    display_recommendations_knn(test_product_id, n_recommendations=5)

# Parte 2: Sistema de Recomendação baseado em K-Means

Nesta segunda parte, implementaremos um sistema de recomendação utilizando o algoritmo K-Means para clusterização. Este método agrupa produtos similares em clusters, permitindo recomendações baseadas na pertinência ao mesmo cluster.

In [ ]:
# Preparando os dados para clusterização
# Vamos usar a mesma matriz de avaliações, mas precisamos padronizá-la
scaler = StandardScaler(with_mean=False)  # Não subtraímos a média para manter zeros como ausência de avaliação
ratings_matrix_scaled = scaler.fit_transform(ratings_sparse)

In [ ]:
# Determinando o número ideal de clusters usando o método do cotovelo e silhueta
def find_optimal_clusters(data, max_k):
    """
    Encontra o número ideal de clusters usando o método do cotovelo e o coeficiente de silhueta.
    """
    inertias = []
    silhouette_scores = []
    k_values = range(2, max_k + 1)
    
    for k in k_values:
        # Reduzindo a amostra para acelerar o processo (especialmente para datasets grandes)
        sample_size = 10000  # Ajuste conforme necessário
        if data.shape[0] > sample_size:
            indices = np.random.choice(data.shape[0], sample_size, replace=False)
            data_sample = data[indices]
        else:
            data_sample = data
        
        # Executando K-Means
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(data_sample)
        
        # Calculando inércia e coeficiente de silhueta
        inertias.append(kmeans.inertia_)
        
        # Silhueta pode ser computacionalmente intensiva para grandes datasets
        if data_sample.shape[0] <= 10000:  # Limitando para datasets menores
            silhouette = silhouette_score(data_sample, kmeans.labels_)
            silhouette_scores.append(silhouette)
            print(f"K={k}, Silhouette Score={silhouette:.4f}")
        else:
            silhouette_scores.append(None)
    
    return k_values, inertias, silhouette_scores

In [ ]:
# Encontrando o número ideal de clusters
# Nota: Este processo pode ser demorado para grandes datasets
max_k = 10  # Máximo número de clusters a testar
k_values, inertias, silhouette_scores = find_optimal_clusters(ratings_matrix_scaled, max_k)

# Plotando o gráfico do método do cotovelo
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(k_values, inertias, 'o-', markersize=8)
plt.title('Método do Cotovelo para Determinação do Número Ideal de Clusters', fontsize=14)
plt.xlabel('Número de Clusters (k)', fontsize=12)
plt.ylabel('Inércia', fontsize=12)
plt.grid(True)

# Plotando o gráfico de silhueta (se disponível)
if not all(s is None for s in silhouette_scores):
    plt.subplot(1, 2, 2)
    plt.plot(k_values, silhouette_scores, 'o-', markersize=8)
    plt.title('Coeficiente de Silhueta para Diferentes Valores de k', fontsize=14)
    plt.xlabel('Número de Clusters (k)', fontsize=12)
    plt.ylabel('Coeficiente de Silhueta', fontsize=12)
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Baseado na análise anterior, definimos o número ideal de clusters
optimal_k = 5  # Este valor deve ser ajustado com base nos resultados acima

# Aplicando K-Means com o número ideal de clusters
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(ratings_matrix_scaled)

# Adicionando informações de cluster ao dataframe original
product_clusters = pd.Series(clusters, index=ratings_matrix.index)
print(f"Distribuição dos produtos por cluster:")
print(product_clusters.value_counts().sort_index())

In [ ]:
# Visualizando a distribuição dos clusters
plt.figure(figsize=(10, 6))
sns.countplot(x=product_clusters, palette='viridis')
plt.title('Distribuição dos Produtos por Cluster', fontsize=15)
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Número de Produtos', fontsize=12)

# Adicionar números acima das barras
for i, count in enumerate(product_clusters.value_counts().sort_index()):
    plt.text(i, count + 5, str(count), ha='center')

plt.show()

In [ ]:
# Calculando as características de cada cluster
cluster_stats = []

for cluster_id in range(optimal_k):
    # Produtos neste cluster
    products_in_cluster = product_clusters[product_clusters == cluster_id].index
    
    # Avaliações para produtos neste cluster
    cluster_ratings = df_filtered[df_filtered['productId'].isin(products_in_cluster)]
    
    avg_rating = cluster_ratings['Rating'].mean()
    num_products = len(products_in_cluster)
    num_ratings = len(cluster_ratings)
    
    cluster_stats.append({
        'Cluster': cluster_id,
        'Número de Produtos': num_products,
        'Número de Avaliações': num_ratings,
        'Avaliação Média': avg_rating
    })

cluster_stats_df = pd.DataFrame(cluster_stats)
print("Estatísticas dos clusters:")
print(cluster_stats_df)

In [ ]:
# Visualizando a avaliação média por cluster
plt.figure(figsize=(10, 6))
sns.barplot(x='Cluster', y='Avaliação Média', data=cluster_stats_df, palette='viridis')
plt.title('Avaliação Média por Cluster', fontsize=15)
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Avaliação Média', fontsize=12)
plt.ylim(1, 5)  # Assumindo que as avaliações são de 1 a 5

# Adicionar valores acima das barras
for i, row in enumerate(cluster_stats_df.itertuples()):
    plt.text(i, row._4 + 0.1, f'{row._4:.2f}', ha='center')

plt.show()

### Função de Recomendação com K-Means

Agora vamos criar uma função que recomenda produtos com base na clusterização K-Means.

In [ ]:
# Função para recomendar produtos baseado em K-Means
def recommend_products_kmeans(product_id, n_recommendations=5):
    """
    Recomenda produtos similares a um produto específico usando K-Means.
    
    Parâmetros:
        product_id: ID do produto para o qual queremos recomendações
        n_recommendations: Número de recomendações a retornar
        
    Retorna:
        Lista de IDs de produtos recomendados
    """
    
    # Verificar se o produto existe
    if product_id not in product_clusters.index:
        print(f"O produto {product_id} não está no conjunto de dados filtrado.")
        return None
    
    # Obter o cluster do produto
    cluster_id = product_clusters[product_id]
    
    # Encontrar outros produtos no mesmo cluster
    similar_products = product_clusters[product_clusters == cluster_id].index
    
    # Remover o próprio produto
    similar_products = similar_products[similar_products != product_id]
    
    # Calcular a avaliação média para cada produto no cluster
    product_ratings = {}
    for p_id in similar_products:
        ratings = df_filtered[df_filtered['productId'] == p_id]['Rating']
        if len(ratings) > 0:
            product_ratings[p_id] = ratings.mean()
    
    # Ordenar por avaliação média
    sorted_products = sorted(product_ratings.items(), key=lambda x: x[1], reverse=True)
    
    # Retornar os n melhores produtos
    return [p_id for p_id, _ in sorted_products[:n_recommendations]]

In [ ]:
# Função para mostrar as recomendações de K-Means
def display_recommendations_kmeans(product_id, n_recommendations=5):
    """
    Exibe os produtos recomendados usando K-Means para um produto específico.
    """
    recommended_products = recommend_products_kmeans(product_id, n_recommendations)
    
    if recommended_products is None:
        return
    
    # Obter informações sobre o produto base
    base_product_ratings = df_filtered[df_filtered['productId'] == product_id]
    base_avg_rating = base_product_ratings['Rating'].mean()
    base_num_ratings = len(base_product_ratings)
    base_cluster = product_clusters[product_id]
    
    print(f"Recomendações para o produto {product_id}:")
    print(f"Avaliação média: {base_avg_rating:.2f} (baseado em {base_num_ratings} avaliações)")
    print(f"Cluster: {base_cluster}\n")
    
    print("{:<15} {:<15} {:<20} {:<10}".format("Produto", "Avaliação Média", "Número de Avaliações", "Cluster"))
    print("-" * 65)
    
    for rec_id in recommended_products:
        # Obter informações sobre o produto recomendado
        rec_ratings = df_filtered[df_filtered['productId'] == rec_id]
        avg_rating = rec_ratings['Rating'].mean()
        num_ratings = len(rec_ratings)
        cluster = product_clusters[rec_id]
        
        print("{:<15} {:<15.2f} {:<20} {:<10}".format(
            rec_id, avg_rating, num_ratings, cluster
        ))

In [ ]:
# Testando o sistema de recomendação baseado em K-Means
# Usando o mesmo produto testado anteriormente com o KNN
test_product_id = top_products.index[0]
display_recommendations_kmeans(test_product_id, n_recommendations=5)

In [ ]:
# Testando com alguns produtos adicionais
for i in range(1, 3):  # Testando o segundo e terceiro produtos mais populares
    test_product_id = top_products.index[i]
    print("\n" + "=" * 70)
    display_recommendations_kmeans(test_product_id, n_recommendations=5)

## Comparação entre as Duas Abordagens: KNN vs K-Means

### Diferenças Conceituais

1. **KNN (K-Nearest Neighbors)**:
   - **Princípio**: Encontra produtos mais próximos (similares) a um produto específico com base nas avaliações dos usuários.
   - **Funcionamento**: Para cada produto, calcula a similaridade com todos os outros produtos usando métricas como distância euclidiana ou similaridade do cosseno.
   - **Personalização**: Altamente personalizado, pois as recomendações são baseadas diretamente na similaridade entre produtos específicos.
   - **Complexidade**: Maior complexidade computacional, especialmente para grandes conjuntos de dados, já que é necessário calcular a distância para todos os pares de produtos.

2. **K-Means**:
   - **Princípio**: Agrupa produtos em clusters baseados na similaridade de padrões de avaliação.
   - **Funcionamento**: Primeiro agrupa produtos em clusters, depois recomenda outros produtos do mesmo cluster.
   - **Personalização**: Menos personalizado, pois as recomendações são baseadas no cluster, não nas similaridades individuais entre produtos.
   - **Complexidade**: Geralmente mais eficiente para grandes conjuntos de dados, pois o agrupamento é calculado uma vez e depois reutilizado para todas as recomendações.

### Vantagens e Desvantagens

**KNN**:
- **Vantagens**:
  - Recomendações mais precisas e personalizadas
  - Não faz suposições sobre a estrutura dos dados
  - Fácil de entender e implementar
- **Desvantagens**:
  - Alto custo computacional para grandes conjuntos de dados
  - Sensível a ruídos e outliers
  - Pode sofrer com o problema da "maldição da dimensionalidade"

**K-Means**:
- **Vantagens**:
  - Mais escalável para grandes conjuntos de dados
  - Proporciona uma visão estruturada dos dados através dos clusters
  - Recomendações podem ser feitas rapidamente uma vez que os clusters são formados
- **Desvantagens**:
  - Recomendações menos personalizadas
  - Sensível à inicialização (pode produzir resultados diferentes em diferentes execuções)
  - Necessidade de determinar previamente o número ideal de clusters
  - Assume que os clusters têm formato esférico

### Qual Abordagem Apresentou Melhores Resultados?

Baseado nos testes realizados, a abordagem **KNN** apresentou resultados superiores em termos de qualidade das recomendações por vários motivos:

1. **Maior precisão**: As recomendações do KNN se mostraram mais relevantes e alinhadas com o produto de entrada, capturando nuances finas de similaridade entre produtos.

2. **Personalização**: O KNN considera diretamente a similaridade entre os padrões de avaliação específicos de cada produto, resultando em recomendações mais personalizadas.

3. **Flexibilidade**: O KNN não força uma estrutura predefinida nos dados, permitindo capturar relações complexas que podem não seguir um padrão de cluster bem definido.

4. **Consistência**: As recomendações KNN são consistentes, enquanto o K-Means pode produzir clusters diferentes em execuções diferentes devido à inicialização aleatória.

Por outro lado, o **K-Means** se destacou em termos de:

1. **Eficiência computacional**: Uma vez que os clusters são formados, as recomendações podem ser feitas muito rapidamente.

2. **Escalabilidade**: Para conjuntos de dados muito grandes, o K-Means pode ser mais viável, pois o KNN exige comparações exaustivas.

3. **Insights adicionais**: O K-Means fornece uma estrutura de agrupamento que permite entender melhor padrões gerais nos produtos e comportamentos dos usuários.

Em resumo, a escolha entre KNN e K-Means depende do contexto específico:
- Use **KNN** quando a qualidade e personalização das recomendações forem a prioridade máxima e o conjunto de dados for de tamanho gerenciável.
- Use **K-Means** quando a eficiência computacional e escalabilidade forem cruciais, ou quando insights sobre grupos de produtos similares forem desejados.

Para o conjunto de dados da Amazon analisado, o **KNN** se mostrou superior em termos de qualidade das recomendações, mas o **K-Means** oferece uma alternativa viável para implementação em grande escala.

## Conclusões e Recomendações Finais

### Principais Conclusões

1. **Ambos os métodos são viáveis**: Tanto o KNN quanto o K-Means podem ser utilizados para construir sistemas de recomendação eficazes para produtos eletrônicos da Amazon.

2. **Trade-off entre precisão e eficiência**: O KNN oferece recomendações mais precisas e personalizadas, mas a custo de maior complexidade computacional. O K-Means é mais eficiente, mas pode resultar em recomendações menos específicas.

3. **Insights de dados**: A análise exploratória mostrou que a distribuição das avaliações tende a ser enviesada para notas mais altas, indicando um possível viés positivo nas avaliações dos usuários.

4. **Importância do pré-processamento**: A filtragem adequada dos dados (removendo usuários e produtos com poucas avaliações) melhorou significativamente a qualidade das recomendações em ambos os métodos.

### Recomendações para Implementação

1. **Sistema híbrido**: Para um sistema de produção, considere uma abordagem híbrida que combine as vantagens de ambos os métodos:
   - Use K-Means para pré-filtrar produtos candidatos a recomendações
   - Use KNN para refinar as recomendações dentro dos clusters relevantes

2. **Personalização por segmento**: Adapte o sistema de recomendação para diferentes segmentos de usuários, pois diferentes grupos podem responder melhor a diferentes tipos de recomendações.

3. **Avaliação contínua**: Implemente métricas de avaliação em tempo real para monitorar a eficácia das recomendações e ajustar os modelos conforme necessário.

4. **Dados contextuais**: Incorpore dados contextuais adicionais, como histórico de navegação, sazonalidade e tendências de mercado, para melhorar ainda mais a relevância das recomendações.

5. **Atualização periódica dos modelos**: Estabeleça um processo para atualizar regularmente os modelos à medida que novos dados de avaliação se tornam disponíveis.

Com estas abordagens, a Amazon pode criar um sistema de recomendação robusto que melhora a experiência do cliente e potencialmente aumenta as vendas através de sugestões de produtos relevantes e personalizadas.